In [ ]:
Análisis Efecto de salidas en Índices

SyntaxError: invalid syntax (<ipython-input-5-3e5ae96ee00e>, line 1)

In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

In [3]:
# Ruta del archivo CSV
addedremovestocks_csv = '/content/sp500changes.csv'

# Cargar el archivo CSV como un DataFrame
df = pd.read_csv(addedremovestocks_csv, names=['fecha', 'tickeradded', 'tickerremoved'], skiprows=1,delimiter=';')
df['fecha'] = pd.to_datetime(df['fecha'], format='%B %d %Y')

# Muestra las primeras filas del DataFrame para verificar su contenido
print(df.head())
print(df.iloc[3,1])

FileNotFoundError: [Errno 2] No such file or directory: '/content/sp500changes.csv'

In [ ]:

def get_prices(symbol, start_date_str, ticker_type):
    API_KEY = 'MWIJ6VEA5NEXK0ET'
    function = 'TIME_SERIES_DAILY'
    outputsize = 'full'  # Usar 'full' para asegurarse de obtener suficientes datos históricos

    # Construye la URL de la solicitud
    url = f'https://www.alphavantage.co/query?function={function}&symbol={symbol}&outputsize={outputsize}&apikey={API_KEY}'

    # Realiza la solicitud a la API
    response = requests.get(url)
    data = response.json()
    # Verifica si la respuesta contiene la clave 'Time Series (Daily)'
    if 'Time Series (Daily)' in data:
        # Extrae los datos de precios y convierte a DataFrame
        df = pd.DataFrame(data['Time Series (Daily)']).T
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

        # Convierte los índices en formato de fecha y los datos a números flotantes
        df.index = pd.to_datetime(df.index)
        for col in df.columns:
            df[col] = df[col].astype(float)

        # Convierte la fecha de inicio de string a datetime
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')

        # Filtra el DataFrame para el rango de fechas deseado (10 días después de la fecha de inicio)
        end_date = start_date + timedelta(days=180)
        df_filtered = df[(df.index >= start_date) & (df.index <= end_date)]

        # Calcula la variación de la acción
        if not df_filtered.empty:
            opening_price = df_filtered.iloc[0]['Open']
            closing_price = df_filtered.iloc[-1]['Close']
            variation = ((closing_price - opening_price) / opening_price) * 100
            print(f"Variación del {ticker_type} {symbol} desde {start_date.date()} a {end_date.date()}: {variation:.2f}%")
        else:
            print(f"No se encontraron datos para el {ticker_type} {symbol} en el rango desde {start_date.date()} a {end_date.date()}.")
            variation = None

        # Retorna el DataFrame filtrado y la variación
        return df_filtered, variation
    else:
        print(f"Error al obtener datos para {symbol}. Verifica tu clave API y el símbolo del ticker.")
        return None, None

In [ ]:
def process_stock_changes(df):
    # Preparar una lista para almacenar los resultados
    results = []

    for index, row in df.iterrows():
        # Extraer la fecha, el ticker añadido y el ticker eliminado de la fila actual
        date_str = row['fecha'].strftime('%Y-%m-%d')
        ticker_added = row['tickeradded']
        ticker_removed = row['tickerremoved']

        # Procesar el ticker añadido si existe
        if pd.notna(ticker_added):
            # Llama a get_prices con el argumento 'Añadido' para ticker_type
            _, variation_added = get_prices(ticker_added, date_str, 'Añadido')
            # Agregar la variación del ticker añadido a la lista de resultados
            results.append({
                'fecha': date_str,
                'tipo': 'Añadido',
                'ticker': ticker_added,
                'variación': variation_added
            })
            # Espera antes de realizar la próxima solicitud
            time.sleep(2)  # Espera 10 segundos

        # Procesar el ticker eliminado si existe
        if pd.notna(ticker_removed):
            # Llama a get_prices con el argumento 'Eliminado' para ticker_type
            _, variation_removed = get_prices(ticker_removed, date_str, 'Eliminado')
            # Agregar la variación del ticker eliminado a la lista de resultados
            results.append({
                'fecha': date_str,
                'tipo': 'Eliminado',
                'ticker': ticker_removed,
                'variación': variation_removed
            })
            # Espera antes de realizar la próxima solicitud
            time.sleep(2)  # Espera 10 segundos

    # Convertir la lista de resultados en un DataFrame
    results_df = pd.DataFrame(results)

    return results_df
# Aplicar la función al DataFrame y mostrar los resultados
results_df = process_stock_changes(df)
print(results_df)

Variación del Añadido UBER desde 2023-12-18 a 2024-06-15: -22.35%
Variación del Eliminado SEE desde 2023-12-18 a 2024-06-15: 4.20%
Variación del Añadido JBL desde 2023-12-18 a 2024-06-15: -10.13%
Variación del Eliminado ALK desde 2023-12-18 a 2024-06-15: 5.08%
Variación del Añadido BLDR desde 2023-12-18 a 2024-06-15: -17.92%
Variación del Eliminado SEDG desde 2023-12-18 a 2024-06-15: 38.35%
Variación del Añadido HUBB desde 2023-10-18 a 2024-04-15: -23.82%
Variación del Eliminado OGN desde 2023-10-18 a 2024-04-15: -6.91%
Variación del Añadido LULU desde 2023-10-18 a 2024-04-15: -12.10%
No se encontraron datos para el Eliminado ATVI en el rango desde 2023-10-18 a 2024-04-15.
Variación del Eliminado DXC desde 2023-10-03 a 2024-03-31: -3.29%
Variación del Añadido VLTO desde 2023-10-02 a 2024-03-30: -9.22%
Variación del Añadido BX desde 2023-09-18 a 2024-03-16: -10.52%
Variación del Eliminado LNC desde 2023-09-18 a 2024-03-16: -5.42%
Variación del Añadido ABNB desde 2023-09-18 a 2024-03-16:

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:


# Filtra las filas para tickers añadidos y calcula el promedio de sus variaciones
variaciones_añadidas = results_df[results_df['tipo'] == 'Añadido']['variación']
promedio_añadidos = variaciones_añadidas.mean()

# Filtra las filas para tickers eliminados y calcula el promedio de sus variaciones
variaciones_eliminadas = results_df[results_df['tipo'] == 'Eliminado']['variación']
promedio_eliminados = variaciones_eliminadas.mean()

# Imprime los resultados
print(f"Promedio de variaciones para tickers añadidos: {promedio_añadidos}")
print(f"Promedio de variaciones para tickers eliminados: {promedio_eliminados}")


Promedio de variaciones para tickers añadidos: 0.7168739116131075
Promedio de variaciones para tickers eliminados: 1.8583522390630602


In [ ]:
results_df.to_csv('/content/dif5days.csv', index=False)